Prepare data for models

In [65]:
import os
import librosa
import numpy as np
from tqdm import tqdm

DATA_PATH = '../Data/speech_commands/'
SAVE_PATH = '../Data/'

TARGET_WORDS = ['on', 'off', 'stop']
UNKNOWN_WORDS = ['right', 'left', 'up', 'down']

X = [] 
y = [] 

def get_mfcc(file_path):
    try:
        # 1. Load audio file
        audio, sr = librosa.load(file_path, sr=16000)
        
        # 2. FEATURE ENGINEERING: Trim silence from beginning and end
        # top_db=25 means any sound 25dB below the peak volume is considered silence and removed
        trimmed_audio, _ = librosa.effects.trim(audio, top_db=25)
        
        # Filter out extremely short noises (e.g., clicks shorter than 0.15 seconds)
        if len(trimmed_audio) < 2400: 
            return None

        # 3. Extract MFCC on the PURE spoken word
        mfccs = librosa.feature.mfcc(y=trimmed_audio, sr=sr, n_mfcc=13, hop_length=512)
        mfccs = mfccs.T # Transpose to shape (frames, 13)

        # 4. FEATURE ENGINEERING: Dynamic Temporal Pooling
        # Dynamically split the frames into 3 equal sections, regardless of length
        parts = np.array_split(mfccs, 3)
        
        # Calculate the mean for each dynamic part
        part1 = np.mean(parts[0], axis=0) # Always the start of the word
        part2 = np.mean(parts[1], axis=0) # Always the middle
        part3 = np.mean(parts[2], axis=0) # Always the end

        # Concatenate them into a single 1D array of 39 features
        features = np.concatenate((part1, part2, part3))
        
        return features
    except Exception as e:
        return None

print("Processing Target Words (with Silence Trimming)...")
for label_idx, word in enumerate(TARGET_WORDS):
    word_dir = os.path.join(DATA_PATH, word)
    if not os.path.exists(word_dir):
        print(f"Warning: Directory for '{word}' not found.")
        continue
        
    files = [f for f in os.listdir(word_dir) if f.endswith('.wav')]
    for file_name in tqdm(files[:1500], desc=f"Loading {word}"): 
        file_path = os.path.join(word_dir, file_name)
        mfcc_data = get_mfcc(file_path)
        if mfcc_data is not None:
            X.append(mfcc_data)
            y.append(label_idx)

print("\nProcessing Unknown Words (with Silence Trimming)...")
unknown_label_idx = len(TARGET_WORDS) 
for word in UNKNOWN_WORDS:
    word_dir = os.path.join(DATA_PATH, word)
    if not os.path.exists(word_dir):
        continue
        
    files = [f for f in os.listdir(word_dir) if f.endswith('.wav')]
    for file_name in tqdm(files[:375], desc=f"Loading {word} (Unknown)"): 
        file_path = os.path.join(word_dir, file_name)
        mfcc_data = get_mfcc(file_path)
        if mfcc_data is not None:
            X.append(mfcc_data)
            y.append(unknown_label_idx)

X = np.array(X)
y = np.array(y)

np.save(os.path.join(SAVE_PATH, 'X_features.npy'), X)
np.save(os.path.join(SAVE_PATH, 'y_labels.npy'), y)

print(f"\nFeature extraction complete! Saved {len(X)} robust samples.")
print(f"NEW Features shape: {X.shape} (Dynamic Temporal Pooling)")

Processing Target Words (with Silence Trimming)...


Loading stop: 100%|██████████| 1500/1500 [00:07<00:00, 200.58it/s]



Processing Unknown Words (with Silence Trimming)...


Loading down (Unknown): 100%|██████████| 375/375 [00:02<00:00, 151.64it/s]


Feature extraction complete! Saved 5995 robust samples.
NEW Features shape: (5995, 39) (Dynamic Temporal Pooling)


RandomForest

In [44]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
# import m2cgen as m2c
import os

DATA_PATH = '../Data/'
OUTPUT_PATH = '../Output'

# 1. Load the pre-processed features and labels
X = np.load(os.path.join(DATA_PATH, 'X_features.npy'))
y = np.load(os.path.join(DATA_PATH, 'y_labels.npy'))

print(f"Loaded Data -> X shape: {X.shape}, y shape: {y.shape}")

# Split the dataset into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Define the Optuna objective function for a TINY Random Forest model
def objective(trial):
    param = {
        # STRICT LIMITS: Keeping trees small for the MCU's RAM
        'n_estimators': trial.suggest_int('n_estimators', 10, 30), 
        'max_depth': trial.suggest_int('max_depth', 2, 10), 
        'random_state': 42,
        'n_jobs': -1
    }
    
    model = RandomForestClassifier(**param)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    return accuracy_score(y_test, preds)

# 3. Run the hyperparameter optimization
print("\nStarting Optuna optimization for a lightweight model...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print("\nBest parameters found:", study.best_params)
print(f"Best cross-validation accuracy: {study.best_value * 100:.2f}%")

# 4. Train the final Random Forest model
print("\nTraining the final Random Forest model...")
final_model = RandomForestClassifier(**study.best_params, random_state=42)
final_model.fit(X_train, y_train)

# Evaluate the final model
final_preds = final_model.predict(X_test)
print("\nClassification Report on Test Data:")

unique_classes = np.unique(y_test)
target_names = ['on', 'off', 'stop', 'unknown'][:len(unique_classes)]

print(classification_report(y_test, final_preds, target_names=target_names))

# 5. Export the trained model parameters as lightweight arrays for MicroPython
print("\nExporting tree parameters (arrays) for memory-efficient Edge Deployment...")

trees_dump = []
for estimator in final_model.estimators_:
    tree = estimator.tree_
    
    # In Random Forest, tree.value stores the class counts at each node
    # We convert this to the actual predicted class index for leaf nodes
    node_classes = [int(np.argmax(val)) for val in tree.value]
    
    tree_data = {
        'left': tree.children_left.tolist(),
        'right': tree.children_right.tolist(),
        'feature': tree.feature.tolist(),
        # Round thresholds to 4 decimals to save storage space
        'threshold': [round(t, 4) for t in tree.threshold.tolist()], 
        'classes': node_classes
    }
    trees_dump.append(tree_data)

# Save to a Python file as a simple list of dictionaries
output_file = os.path.join(OUTPUT_PATH, 'model_data_randomForest.py')
with open(output_file, 'w') as f:
    f.write("# Auto-generated Random Forest parameters for Edge AI\n")
    f.write("TREES = [\n")
    for t in trees_dump:
        f.write(f"    {t},\n")
    f.write("]\n")

print(f"Model parameters successfully exported to: {output_file}")

[I 2026-06-28 16:24:46,449] A new study created in memory with name: no-name-419168bc-02b3-4029-842b-e7ef0964247b
[I 2026-06-28 16:24:46,614] Trial 0 finished with value: 0.7881567973311092 and parameters: {'n_estimators': 30, 'max_depth': 9}. Best is trial 0 with value: 0.7881567973311092.


Loaded Data -> X shape: (5995, 39), y shape: (5995,)

Starting Optuna optimization for a lightweight model...


[I 2026-06-28 16:24:46,702] Trial 1 finished with value: 0.7489574645537949 and parameters: {'n_estimators': 10, 'max_depth': 9}. Best is trial 0 with value: 0.7881567973311092.
[I 2026-06-28 16:24:46,881] Trial 2 finished with value: 0.7848206839032527 and parameters: {'n_estimators': 27, 'max_depth': 10}. Best is trial 0 with value: 0.7881567973311092.
[I 2026-06-28 16:24:47,003] Trial 3 finished with value: 0.7514595496246872 and parameters: {'n_estimators': 23, 'max_depth': 7}. Best is trial 0 with value: 0.7881567973311092.
[I 2026-06-28 16:24:47,098] Trial 4 finished with value: 0.6630525437864887 and parameters: {'n_estimators': 22, 'max_depth': 2}. Best is trial 0 with value: 0.7881567973311092.
[I 2026-06-28 16:24:47,202] Trial 5 finished with value: 0.6630525437864887 and parameters: {'n_estimators': 23, 'max_depth': 2}. Best is trial 0 with value: 0.7881567973311092.
[I 2026-06-28 16:24:47,297] Trial 6 finished with value: 0.7731442869057548 and parameters: {'n_estimators': 


Best parameters found: {'n_estimators': 28, 'max_depth': 9}
Best cross-validation accuracy: 79.15%

Training the final Random Forest model...

Classification Report on Test Data:
              precision    recall  f1-score   support

          on       0.78      0.79      0.78       299
         off       0.80      0.80      0.80       300
        stop       0.90      0.84      0.87       300
     unknown       0.70      0.74      0.72       300

    accuracy                           0.79      1199
   macro avg       0.79      0.79      0.79      1199
weighted avg       0.79      0.79      0.79      1199


Exporting tree parameters (arrays) for memory-efficient Edge Deployment...
Model parameters successfully exported to: ../Output\model_data_randomForest.py


LogisticRegression

In [45]:
import numpy as np
from sklearn.linear_model import LogisticRegression
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import os

DATA_PATH = '../Data/'
OUTPUT_PATH = '../Output'

# 1. Load the pre-processed features and labels
X = np.load(os.path.join(DATA_PATH, 'X_features.npy'))
y = np.load(os.path.join(DATA_PATH, 'y_labels.npy'))

print(f"Loaded Data -> X shape: {X.shape}, y shape: {y.shape}")

# Split the dataset into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Define the Optuna objective function for Logistic Regression
def objective(trial):
    param = {
        # 'C' is the inverse of regularization strength. 
        # Smaller values specify stronger limits/constraints on the weights.
        'C': trial.suggest_float('C', 1e-4, 1e2, log=True),
        
        # 'solver' is the algorithm used to optimize the weights
        'solver': trial.suggest_categorical('solver', ['lbfgs', 'newton-cg', 'saga']),
        
        # Max_iter is kept high to allow the mathematical solver to fully converge
        'max_iter': 2000, 
        'random_state': 42,
        'n_jobs': -1
    }
    
    model = LogisticRegression(**param)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    return accuracy_score(y_test, preds)

# 3. Run the hyperparameter optimization
print("\nStarting Optuna optimization for Logistic Regression...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print("\nBest parameters found:", study.best_params)
print(f"Best cross-validation accuracy: {study.best_value * 100:.2f}%")

# 4. Train the final model using the exact optimized parameters
print("\nTraining the final Logistic Regression model...")
final_model = LogisticRegression(**study.best_params, max_iter=2000, random_state=42)
final_model.fit(X_train, y_train)

# Evaluate the final model
final_preds = final_model.predict(X_test)
print("\nClassification Report on Test Data:")

# Dynamically extract classes
unique_classes = np.unique(y_test)
target_names = ['on', 'off', 'stop', 'unknown'][:len(unique_classes)]
print(classification_report(y_test, final_preds, target_names=target_names))

# 5. Export weights and intercepts to a tiny Python file
print("\nExporting model parameters (Weights & Intercepts) for Edge Deployment...")

weights = final_model.coef_.tolist()
intercepts = final_model.intercept_.tolist()

output_file = os.path.join(OUTPUT_PATH, 'model_data_regression.py')
with open(output_file, 'w') as f:
    f.write("# Auto-Generated Lightweight Model for Keyword Spotting\n\n")
    f.write(f"CLASSES = {target_names}\n\n")
    
    # We round the numbers to 4 decimal places to save even more space on the MCU
    f.write("WEIGHTS = [\n")
    for class_weights in weights:
        rounded_weights = [round(w, 4) for w in class_weights]
        f.write(f"    {rounded_weights},\n")
    f.write("]\n\n")
    
    rounded_intercepts = [round(i, 4) for i in intercepts]
    f.write(f"INTERCEPTS = {rounded_intercepts}\n")

print(f"Model successfully exported to: {output_file}")
print("Done! The model_data.py is now highly optimized and incredibly lightweight.")

[I 2026-06-28 16:25:44,984] A new study created in memory with name: no-name-0c975743-a44a-4ddb-8839-808d10d11ce2


Loaded Data -> X shape: (5995, 39), y shape: (5995,)

Starting Optuna optimization for Logistic Regression...


c:\Users\User\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
[I 2026-06-28 16:25:46,756] Trial 0 finished with value: 0.7948290241868223 and parameters: {'C': 16.896513260974356, 'solver': 'saga'}. Best is trial 0 with value: 0.7948290241868223.
c:\Users\User\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
[I 2026-06-28 16:25:48,773] Trial 1 finished with value: 0.7948290241868223 and parameters: {'C': 0.422188412563391, 'solver': 'saga'}. Best is trial 0 with value: 0.7948290241868223.
c:\Users\User\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no 


Best parameters found: {'C': 16.896513260974356, 'solver': 'saga'}
Best cross-validation accuracy: 79.48%

Training the final Logistic Regression model...

Classification Report on Test Data:
              precision    recall  f1-score   support

          on       0.79      0.82      0.80       299
         off       0.82      0.79      0.80       300
        stop       0.88      0.82      0.85       300
     unknown       0.70      0.75      0.73       300

    accuracy                           0.79      1199
   macro avg       0.80      0.79      0.80      1199
weighted avg       0.80      0.79      0.80      1199


Exporting model parameters (Weights & Intercepts) for Edge Deployment...
Model successfully exported to: ../Output\model_data_regression.py
Done! The model_data.py is now highly optimized and incredibly lightweight.


In [47]:
import sys
import os

sys.path.append(os.path.abspath('../Output'))

import model_data_regression as model_data

def predict_audio_class(mfcc_features):
    """
    Computes the dot product of features and weights for each class,
    adds the intercept, and returns the class with the highest score.
    """
    num_classes = len(model_data.INTERCEPTS)
    num_features = len(mfcc_features)
    
    scores = []
    
    # Calculate score for each class
    for i in range(num_classes):
        class_score = model_data.INTERCEPTS[i]
        
        # Dot product: multiply each feature by its corresponding weight
        for j in range(num_features):
            class_score += mfcc_features[j] * model_data.WEIGHTS[i][j]
            
        scores.append(class_score)
        
    # Find the index of the highest score (Argmax)
    best_class_idx = 0
    max_score = scores[0]
    for i in range(1, num_classes):
        if scores[i] > max_score:
            max_score = scores[i]
            best_class_idx = i
            
    return best_class_idx

# --- Example Usage on Board ---
# ['on', 'off', 'stop', 'uknown']
i = 16
features = X_test[i] # 13 features from mic
result_idx = predict_audio_class(features)
print("Predicted Word:", model_data.CLASSES[result_idx], "and actuall is:", y_test[i])

IndexError: list index out of range

XGBoost

In [ ]:
import numpy as np
import xgboost as xgb
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import os
import json

DATA_PATH = '../Data/'
OUTPUT_PATH = '../Output'

# 1. Load the pre-processed features and labels
X = np.load(os.path.join(DATA_PATH, 'X_features.npy'))
y = np.load(os.path.join(DATA_PATH, 'y_labels.npy'))

print(f"Loaded Data -> X shape: {X.shape}, y shape: {y.shape}")

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Define Optuna objective for XGBoost
def objective(trial):
    param = {
        'objective': 'multi:softprob',
        'max_depth': trial.suggest_int('max_depth', 2, 3), 
        'n_estimators': trial.suggest_int('n_estimators', 2, 5), 
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'random_state': 42,
        'n_jobs': -1
    }
    
    model = xgb.XGBClassifier(**param)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    return accuracy_score(y_test, preds)

# 3. Optimize Hyperparameters
print("\nStarting Optuna optimization for XGBoost...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print("\nBest parameters found:", study.best_params)
print(f"Best cross-validation accuracy: {study.best_value * 100:.2f}%")

# 4. Train the Final XGBoost Model
print("\nTraining the final XGBoost model...")
best_params = study.best_params
best_params['objective'] = 'multi:softprob'
best_params['random_state'] = 42

final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X_train, y_train)

# Evaluate the final model
final_preds = final_model.predict(X_test)
print("\nClassification Report on Test Data:")
unique_classes = np.unique(y_test)
target_names = ['on', 'off', 'stop', 'unknown'][:len(unique_classes)]
print(classification_report(y_test, final_preds, target_names=target_names))

# 5. CUSTOM EXPORT: Bypass m2cgen and extract trees directly
print("\nExporting XGBoost trees manually for ultra-lightweight Edge Deployment...")

booster = final_model.get_booster()
# Dump all trees to JSON format
trees_json = booster.get_dump(dump_format='json')

parsed_trees = []
for tree_str in trees_json:
    tree_dict = json.loads(tree_str)
    
    # Recursive function to find the maximum node ID to size our arrays
    def get_max_nodeid(node):
        max_id = node.get('nodeid', 0)
        if 'children' in node:
            for child in node['children']:
                max_id = max(max_id, get_max_nodeid(child))
        return max_id
        
    size = get_max_nodeid(tree_dict) + 1
    
    # Initialize flattened arrays for MicroPython
    features = [-1] * size
    thresholds = [0.0] * size
    lefts = [-1] * size
    rights = [-1] * size
    values = [0.0] * size
    
    # Recursive function to parse JSON into flat arrays
    def parse_node(node):
        nid = node['nodeid']
        if 'leaf' in node:
            values[nid] = round(node['leaf'], 4)
        else:
            # XGBoost features are formatted as 'f0', 'f1', etc. We extract the integer.
            feat_str = node['split']
            features[nid] = int(feat_str.replace('f', ''))
            thresholds[nid] = round(node['split_condition'], 4)
            lefts[nid] = node['yes']
            rights[nid] = node['no']
            for child in node['children']:
                parse_node(child)
                
    parse_node(tree_dict)
    
    parsed_trees.append({
        'features': features,
        'thresholds': thresholds,
        'left': lefts,
        'right': rights,
        'values': values
    })

# Save the parsed structure to a Python file
output_file = os.path.join(OUTPUT_PATH, 'model_data_xgb.py')
with open(output_file, 'w') as f:
    f.write("# Auto-Generated Custom XGBoost Export for MicroPython\n\n")
    f.write(f"CLASSES = {target_names}\n")
    f.write(f"NUM_CLASS = {len(target_names)}\n\n")
    f.write("TREES = [\n")
    for t in parsed_trees:
        f.write(f"    {t},\n")
    f.write("]\n")

print(f"XGBoost Model successfully exported to: {output_file}")

[I 2026-06-28 18:35:50,070] A new study created in memory with name: no-name-1df3ab1a-6db6-4e9b-a963-7c89ba30726c
[I 2026-06-28 18:35:50,195] Trial 0 finished with value: 0.7114261884904087 and parameters: {'max_depth': 3, 'n_estimators': 11, 'learning_rate': 0.18267603982484373}. Best is trial 0 with value: 0.7114261884904087.


Loaded Data -> X shape: (5995, 24), y shape: (5995,)

Starting Optuna optimization for XGBoost...


[I 2026-06-28 18:35:50,292] Trial 1 finished with value: 0.718932443703086 and parameters: {'max_depth': 3, 'n_estimators': 14, 'learning_rate': 0.17428264968762794}. Best is trial 1 with value: 0.718932443703086.
[I 2026-06-28 18:35:50,367] Trial 2 finished with value: 0.7322768974145121 and parameters: {'max_depth': 3, 'n_estimators': 11, 'learning_rate': 0.2711834857097505}. Best is trial 2 with value: 0.7322768974145121.
[I 2026-06-28 18:35:50,433] Trial 3 finished with value: 0.7180984153461217 and parameters: {'max_depth': 2, 'n_estimators': 15, 'learning_rate': 0.2802475241894769}. Best is trial 2 with value: 0.7322768974145121.
[I 2026-06-28 18:35:50,495] Trial 4 finished with value: 0.683069224353628 and parameters: {'max_depth': 3, 'n_estimators': 10, 'learning_rate': 0.13616408143549646}. Best is trial 2 with value: 0.7322768974145121.
[I 2026-06-28 18:35:50,563] Trial 5 finished with value: 0.731442869057548 and parameters: {'max_depth': 3, 'n_estimators': 12, 'learning_rat


Best parameters found: {'max_depth': 3, 'n_estimators': 12, 'learning_rate': 0.24915606338574334}
Best cross-validation accuracy: 73.56%

Training the final XGBoost model...

Classification Report on Test Data:
              precision    recall  f1-score   support

          on       0.69      0.75      0.72       299
         off       0.77      0.77      0.77       300
        stop       0.86      0.73      0.79       300
     unknown       0.65      0.69      0.67       300

    accuracy                           0.74      1199
   macro avg       0.74      0.74      0.74      1199
weighted avg       0.74      0.74      0.74      1199


Exporting XGBoost trees manually for ultra-lightweight Edge Deployment...
XGBoost Model successfully exported to: ../Output\model_data_xgb.py


In [56]:
import sys
import os

sys.path.append(os.path.abspath('../Output'))

import model_data_xgb as model_data
def predict_audio_class(mfcc_features):
    """
    Traverses the exported XGBoost trees using the extracted MFCC features.
    Returns the index of the predicted class based on the highest raw score.
    """
    num_class = model_data.NUM_CLASS
    
    # Array to accumulate scores (raw logits) for each class
    class_scores = [0.0] * num_class
    
    # In multi-class XGBoost, trees are assigned to classes in a round-robin fashion.
    # Tree 0 -> Class 0, Tree 1 -> Class 1 ... Tree 4 -> Class 0, etc.
    for i, tree in enumerate(model_data.TREES):
        node = 0 # Start at the root of the tree
        
        # Traverse until a leaf is reached (marked by feature index -1)
        while tree['features'][node] != -1:
            feat_idx = tree['features'][node]
            
            # XGBoost splitting rule: strictly less than (<)
            if mfcc_features[feat_idx] < tree['thresholds'][node]:
                node = tree['left'][node]
            else:
                node = tree['right'][node]
                
        # Leaf reached: get the value and add it to the corresponding class score
        leaf_value = tree['values'][node]
        class_idx = i % num_class
        class_scores[class_idx] += leaf_value
        
    # Find the index of the highest score (Argmax)
    best_class = 0
    max_score = class_scores[0]
    for i in range(1, num_class):
        if class_scores[i] > max_score:
            max_score = class_scores[i]
            best_class = i
            
    return best_class

# --- Example Usage on Board ---
# ['on', 'off', 'stop', 'uknown']
i = 16
features = [-2.5686844e+02,  1.3959177e+02,  8.0665617e+00, -4.1448826e+01, -2.9944046e+01,  3.2988396e-01,  6.2178426e+00,  9.0643892e+00, -6.5838027e+00,  5.9275532e+00, -2.8192373e+01,  1.9626240e+01, -1.0130860e+00, -2.5470102e+02,  1.6171916e+02, -9.4717093e+00, -1.6660915e+01,  9.5316715e+00,  1.5823692e+01, -4.0831656e+00, -1.6109245e+01, -1.6403475e+01, -2.7065601e+00, -7.6615257e+00, 1.2409652e+01, -2.4190521e+01, -4.9443311e+02,  8.9877983e+01, 1.8117041e+01,  3.1036968e+01,  5.0818684e+01,  3.1483164e+01, 4.5603323e+00, -1.2117879e+01, -1.4944232e+01,  2.7241716e+00, 7.9998846e+00, -1.2753278e+01, -1.6924435e+01] # 13 features from mic
result_idx = predict_audio_class(features)
print("Predicted Word:", model_data.CLASSES[result_idx], "and actuall is:", y_test[i])


Predicted Word: off and actuall is: 1


*now with Root reduction and feature importance*

In [58]:
import os
import librosa
import numpy as np
from tqdm import tqdm

DATA_PATH = '../Data/speech_commands/'
SAVE_PATH = '../Data/'

TARGET_WORDS = ['on', 'off', 'stop']
UNKNOWN_WORDS = ['right', 'left', 'up', 'down']

X = [] 
y = [] 

def get_mfcc(file_path):
    try:
        audio, sr = librosa.load(file_path, sr=16000)
        
        # Trim silence
        trimmed_audio, _ = librosa.effects.trim(audio, top_db=25)
        if len(trimmed_audio) < 2400: 
            return None

        # ROOT REDUCTION: Extract only 8 MFCCs instead of 13
        mfccs = librosa.feature.mfcc(y=trimmed_audio, sr=sr, n_mfcc=8, hop_length=512)
        mfccs = mfccs.T 

        # Dynamic Temporal Pooling (3 parts)
        parts = np.array_split(mfccs, 3)
        
        part1 = np.mean(parts[0], axis=0) 
        part2 = np.mean(parts[1], axis=0) 
        part3 = np.mean(parts[2], axis=0) 

        # Concatenate: 8 + 8 + 8 = 24 features
        features = np.concatenate((part1, part2, part3))
        
        return features
    except Exception as e:
        return None

print("Processing Target Words (with 8-MFCC Root Reduction)...")
for label_idx, word in enumerate(TARGET_WORDS):
    word_dir = os.path.join(DATA_PATH, word)
    if not os.path.exists(word_dir):
        continue
        
    files = [f for f in os.listdir(word_dir) if f.endswith('.wav')]
    for file_name in tqdm(files[:1500], desc=f"Loading {word}"): 
        file_path = os.path.join(word_dir, file_name)
        mfcc_data = get_mfcc(file_path)
        if mfcc_data is not None:
            X.append(mfcc_data)
            y.append(label_idx)

print("\nProcessing Unknown Words...")
unknown_label_idx = len(TARGET_WORDS) 
for word in UNKNOWN_WORDS:
    word_dir = os.path.join(DATA_PATH, word)
    if not os.path.exists(word_dir):
        continue
        
    files = [f for f in os.listdir(word_dir) if f.endswith('.wav')]
    for file_name in tqdm(files[:375], desc=f"Loading {word} (Unknown)"): 
        file_path = os.path.join(word_dir, file_name)
        mfcc_data = get_mfcc(file_path)
        if mfcc_data is not None:
            X.append(mfcc_data)
            y.append(unknown_label_idx)

X = np.array(X)
y = np.array(y)

np.save(os.path.join(SAVE_PATH, 'X_features.npy'), X)
np.save(os.path.join(SAVE_PATH, 'y_labels.npy'), y)

print(f"\nFeature extraction complete! Saved {len(X)} samples.")
print(f"NEW Features shape: {X.shape} (24 features per sample)")

Processing Target Words (with 8-MFCC Root Reduction)...


Loading stop: 100%|██████████| 1500/1500 [00:17<00:00, 86.91it/s] 



Processing Unknown Words...


Loading down (Unknown): 100%|██████████| 375/375 [00:04<00:00, 81.85it/s]


Feature extraction complete! Saved 5995 samples.
NEW Features shape: (5995, 24) (24 features per sample)


In [63]:
import numpy as np
import xgboost as xgb
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import os
import json

DATA_PATH = '../Data/'
OUTPUT_PATH = '../Output'

# 1. Load Data
X = np.load(os.path.join(DATA_PATH, 'X_features.npy'))
y = np.load(os.path.join(DATA_PATH, 'y_labels.npy'))
print(f"Original Data -> X shape: {X.shape}, y shape: {y.shape}")

X_train_full, X_test_full, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. FEATURE SELECTION (Method 1)
print("\n--- Running Feature Selection ---")
# Train a temporary, unconstrained model to find feature importance
temp_model = xgb.XGBClassifier(objective='multi:softprob', random_state=42)
temp_model.fit(X_train_full, y_train)

# Get importance scores and select the top 12 features (out of 24)
importances = temp_model.feature_importances_
# argsort sorts ascending, so we take the last 12 elements for the highest scores
top_12_indices = np.argsort(importances)[-12:] 
top_12_indices = np.sort(top_12_indices) # Sort indices to keep the original order

print(f"Selected Top 12 Feature Indices: {top_12_indices.tolist()}")

# Filter the datasets to ONLY include these 12 features
X_train = X_train_full[:, top_12_indices]
X_test = X_test_full[:, top_12_indices]
print(f"Reduced Data -> X_train shape: {X_train.shape}")

# 3. OPTUNA OPTIMIZATION (on the reduced 12-feature dataset)
print("\n--- Starting Optuna optimization on reduced features ---")
def objective(trial):
    param = {
        'objective': 'multi:softprob',
        # Keep trees extremely shallow for Edge RAM
        'max_depth': trial.suggest_int('max_depth', 2, 3), 
        'n_estimators': trial.suggest_int('n_estimators', 2, 5), 
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'random_state': 42,
        'n_jobs': -1
    }
    
    model = xgb.XGBClassifier(**param)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return accuracy_score(y_test, preds)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)
print(f"\nBest Optuna Accuracy: {study.best_value * 100:.2f}%")

# 4. TRAIN FINAL MODEL
print("\n--- Training Final Lightweight XGBoost ---")
best_params = study.best_params
best_params['objective'] = 'multi:softprob'
best_params['random_state'] = 42

final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X_train, y_train)

preds = final_model.predict(X_test)
unique_classes = np.unique(y_test)
target_names = ['on', 'off', 'stop', 'unknown'][:len(unique_classes)]
print("\nFinal Classification Report:")
print(classification_report(y_test, preds, target_names=target_names))

# 5. CUSTOM EXPORT TO PYTHON ARRAYS
print("\n--- Exporting arrays to model_data.py ---")
booster = final_model.get_booster()
trees_json = booster.get_dump(dump_format='json')

parsed_trees = []
for tree_str in trees_json:
    tree_dict = json.loads(tree_str)
    
    def get_max_nodeid(node):
        max_id = node.get('nodeid', 0)
        if 'children' in node:
            for child in node['children']:
                max_id = max(max_id, get_max_nodeid(child))
        return max_id
        
    size = get_max_nodeid(tree_dict) + 1
    features = [-1] * size
    thresholds = [0.0] * size
    lefts = [-1] * size
    rights = [-1] * size
    values = [0.0] * size
    
    def parse_node(node):
        nid = node['nodeid']
        if 'leaf' in node:
            values[nid] = round(node['leaf'], 4)
        else:
            feat_str = node['split']
            features[nid] = int(feat_str.replace('f', ''))
            thresholds[nid] = round(node['split_condition'], 4)
            lefts[nid] = node['yes']
            rights[nid] = node['no']
            for child in node['children']:
                parse_node(child)
                
    parse_node(tree_dict)
    
    parsed_trees.append({
        'features': features,
        'thresholds': thresholds,
        'left': lefts,
        'right': rights,
        'values': values
    })

output_file = os.path.join(OUTPUT_PATH, 'model_data_xgb_new.py')
with open(output_file, 'w') as f:
    f.write("# Auto-Generated Optimized Custom XGBoost Export\n\n")
    f.write(f"CLASSES = {target_names}\n")
    f.write(f"NUM_CLASS = {len(target_names)}\n")
    # We MUST save the selected feature indices so the MCU knows which features to pass to the model!
    f.write(f"SELECTED_FEATURES = {top_12_indices.tolist()}\n\n") 
    f.write("TREES = [\n")
    for t in parsed_trees:
        f.write(f"    {t},\n")
    f.write("]\n")

print(f"Extremely lightweight XGBoost successfully exported to: {output_file}")

Original Data -> X shape: (5995, 24), y shape: (5995,)

--- Running Feature Selection ---


[I 2026-06-28 18:31:16,707] A new study created in memory with name: no-name-ace39d90-1a98-4517-ac67-3e7e4d767813
[I 2026-06-28 18:31:16,720] Trial 0 finished with value: 0.5963302752293578 and parameters: {'max_depth': 2, 'n_estimators': 2, 'learning_rate': 0.17200462706166578}. Best is trial 0 with value: 0.5963302752293578.
[I 2026-06-28 18:31:16,733] Trial 1 finished with value: 0.591326105087573 and parameters: {'max_depth': 2, 'n_estimators': 2, 'learning_rate': 0.05088289009086883}. Best is trial 0 with value: 0.5963302752293578.
[I 2026-06-28 18:31:16,749] Trial 2 finished with value: 0.5946622185154296 and parameters: {'max_depth': 2, 'n_estimators': 3, 'learning_rate': 0.05832981955240293}. Best is trial 0 with value: 0.5963302752293578.
[I 2026-06-28 18:31:16,772] Trial 3 finished with value: 0.6447039199332777 and parameters: {'max_depth': 3, 'n_estimators': 3, 'learning_rate': 0.026262804196618583}. Best is trial 3 with value: 0.6447039199332777.
[I 2026-06-28 18:31:16,806

Selected Top 12 Feature Indices: [1, 3, 4, 6, 9, 11, 14, 15, 17, 19, 20, 22]
Reduced Data -> X_train shape: (4796, 12)

--- Starting Optuna optimization on reduced features ---


[I 2026-06-28 18:31:16,942] Trial 10 finished with value: 0.6547122602168474 and parameters: {'max_depth': 3, 'n_estimators': 4, 'learning_rate': 0.14341836029729105}. Best is trial 10 with value: 0.6547122602168474.
[I 2026-06-28 18:31:16,973] Trial 11 finished with value: 0.6538782318598833 and parameters: {'max_depth': 3, 'n_estimators': 4, 'learning_rate': 0.13716634555794593}. Best is trial 10 with value: 0.6547122602168474.
[I 2026-06-28 18:31:17,003] Trial 12 finished with value: 0.6547122602168474 and parameters: {'max_depth': 3, 'n_estimators': 4, 'learning_rate': 0.13763054844636968}. Best is trial 10 with value: 0.6547122602168474.
[I 2026-06-28 18:31:17,033] Trial 13 finished with value: 0.6663886572143453 and parameters: {'max_depth': 3, 'n_estimators': 4, 'learning_rate': 0.20395514433936232}. Best is trial 13 with value: 0.6663886572143453.
[I 2026-06-28 18:31:17,064] Trial 14 finished with value: 0.664720600500417 and parameters: {'max_depth': 3, 'n_estimators': 4, 'lea


Best Optuna Accuracy: 69.22%

--- Training Final Lightweight XGBoost ---

Final Classification Report:
              precision    recall  f1-score   support

          on       0.66      0.67      0.66       299
         off       0.73      0.76      0.75       300
        stop       0.86      0.70      0.77       300
     unknown       0.56      0.64      0.60       300

    accuracy                           0.69      1199
   macro avg       0.70      0.69      0.70      1199
weighted avg       0.70      0.69      0.70      1199


--- Exporting arrays to model_data.py ---
Extremely lightweight XGBoost successfully exported to: ../Output\model_data_xgb_new.py


*Gaussian Naive Bayes*

In [74]:
import numpy as np
from sklearn.naive_bayes import GaussianNB
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import os

DATA_PATH = '../Data/'

# 1. Load the FULL 39-feature dataset
X = np.load(os.path.join(DATA_PATH, 'X_features.npy'))
y = np.load(os.path.join(DATA_PATH, 'y_labels.npy'))

print(f"Loaded Data -> X shape: {X.shape}, y shape: {y.shape}")

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. OPTUNA OPTIMIZATION FOR NAIVE BAYES
print("\n--- Starting Optuna optimization for var_smoothing ---")

def objective(trial):
    # Suggest a value for var_smoothing on a log scale (from very small to relatively large)
    var_smoothing = trial.suggest_float('var_smoothing', 1e-11, 1e-1, log=True)
    
    gnb = GaussianNB(var_smoothing=var_smoothing)
    gnb.fit(X_train, y_train)
    preds = gnb.predict(X_test)
    
    return accuracy_score(y_test, preds)

study = optuna.create_study(direction='maximize')
# GNB is incredibly fast, so we can easily run 100 trials in seconds!
study.optimize(objective, n_trials=100) 

print(f"\nBest Optuna var_smoothing: {study.best_params['var_smoothing']}")
print(f"Best Accuracy during CV: {study.best_value * 100:.2f}%")

# 3. TRAIN FINAL MODEL WITH BEST PARAMETERS
print("\n--- Training Final Optimized Gaussian Naive Bayes ---")
final_gnb = GaussianNB(**study.best_params)
final_gnb.fit(X_train, y_train)

preds = final_gnb.predict(X_test)
unique_classes = np.unique(y_test)
target_names = ['on', 'off', 'stop', 'unknown'][:len(unique_classes)]

print("\nFinal Classification Report:")
print(classification_report(y_test, preds, target_names=target_names))

# 4. EXPORT TO PURE PYTHON ARRAYS (Same size, higher accuracy!)
print("\n--- Exporting Statistical Parameters to model_data.py ---")

class_priors = final_gnb.class_prior_
if class_priors is None:
    class_counts = np.bincount(y_train)
    class_priors = class_counts / len(y_train)

priors_list = class_priors.tolist()
means_list = final_gnb.theta_.tolist()
vars_list = final_gnb.var_.tolist()

output_file = os.path.join(DATA_PATH, 'model_data.py')
with open(output_file, 'w') as f:
    f.write("# Auto-Generated Optuna-Optimized Gaussian Naive Bayes Export\n\n")
    f.write(f"CLASSES = {target_names}\n")
    
    pure_priors = [round(float(p), 6) for p in priors_list]
    f.write(f"PRIORS = {pure_priors}\n\n")
    
    f.write("MEANS = [\n")
    for class_means in means_list:
        pure_means = [round(float(m), 6) for m in class_means]
        f.write(f"    {pure_means},\n")
    f.write("]\n\n")
    
    # Notice we don't need to manually add epsilon here anymore, 
    # the optimal var_smoothing already handled the math stabilization!
    f.write("VARIANCES = [\n")
    for class_vars in vars_list:
        pure_vars = [round(float(v), 8) for v in class_vars]
        f.write(f"    {pure_vars},\n")
    f.write("]\n")

print(f"Optimized Model successfully exported to: {output_file}")

[I 2026-06-28 19:09:41,772] A new study created in memory with name: no-name-65286da5-33ed-460a-b496-78bd506b18fa
[I 2026-06-28 19:09:41,776] Trial 0 finished with value: 0.7306088407005839 and parameters: {'var_smoothing': 7.361200010330594e-06}. Best is trial 0 with value: 0.7306088407005839.
[I 2026-06-28 19:09:41,780] Trial 1 finished with value: 0.7306088407005839 and parameters: {'var_smoothing': 7.108632513517686e-06}. Best is trial 0 with value: 0.7306088407005839.
[I 2026-06-28 19:09:41,784] Trial 2 finished with value: 0.7306088407005839 and parameters: {'var_smoothing': 7.50643675334007e-05}. Best is trial 0 with value: 0.7306088407005839.
[I 2026-06-28 19:09:41,789] Trial 3 finished with value: 0.7306088407005839 and parameters: {'var_smoothing': 1.3258086557415428e-05}. Best is trial 0 with value: 0.7306088407005839.
[I 2026-06-28 19:09:41,793] Trial 4 finished with value: 0.7306088407005839 and parameters: {'var_smoothing': 1.2448609434133904e-09}. Best is trial 0 with va

Loaded Data -> X shape: (5995, 39), y shape: (5995,)

--- Starting Optuna optimization for var_smoothing ---


[I 2026-06-28 19:09:41,972] Trial 43 finished with value: 0.7281067556296914 and parameters: {'var_smoothing': 0.0012197648983159012}. Best is trial 5 with value: 0.731442869057548.
[I 2026-06-28 19:09:41,978] Trial 44 finished with value: 0.7306088407005839 and parameters: {'var_smoothing': 3.485427678670701e-05}. Best is trial 5 with value: 0.731442869057548.
[I 2026-06-28 19:09:41,983] Trial 45 finished with value: 0.7122602168473728 and parameters: {'var_smoothing': 0.01948354216136053}. Best is trial 5 with value: 0.731442869057548.
[I 2026-06-28 19:09:41,989] Trial 46 finished with value: 0.7306088407005839 and parameters: {'var_smoothing': 1.4429568939833291e-05}. Best is trial 5 with value: 0.731442869057548.
[I 2026-06-28 19:09:41,993] Trial 47 finished with value: 0.731442869057548 and parameters: {'var_smoothing': 0.00010327944073997671}. Best is trial 5 with value: 0.731442869057548.
[I 2026-06-28 19:09:41,998] Trial 48 finished with value: 0.7306088407005839 and parameters


Best Optuna var_smoothing: 0.0001533584341826507
Best Accuracy during CV: 73.14%

--- Training Final Optimized Gaussian Naive Bayes ---

Final Classification Report:
              precision    recall  f1-score   support

          on       0.72      0.74      0.73       299
         off       0.79      0.71      0.75       300
        stop       0.80      0.80      0.80       300
     unknown       0.64      0.67      0.65       300

    accuracy                           0.73      1199
   macro avg       0.73      0.73      0.73      1199
weighted avg       0.73      0.73      0.73      1199


--- Exporting Statistical Parameters to model_data.py ---
Optimized Model successfully exported to: ../Data/model_data.py


In [83]:
import math
import sys
import os

sys.path.append(os.path.abspath('../Output'))

import model_data_gnb as model_data
def predict_audio_class(mfcc_features):
    """
    Computes the Gaussian Log-Probability for each class and returns
    the index of the class with the highest probability.
    """
    best_class = 0
    # Initialize with negative infinity
    max_log_prob = -float('inf') 
    
    num_classes = len(model_data.CLASSES)
    num_features = len(mfcc_features)
    
    for c in range(num_classes):
        # Start with the log of the prior probability for this class
        log_prob = math.log(model_data.PRIORS[c])
        
        # Add the log probability of each feature given the class
        for i in range(num_features):
            x = mfcc_features[i]
            mean = model_data.MEANS[c][i]
            var = model_data.VARIANCES[c][i]
            
            # Gaussian Log-PDF Formula: 
            # -0.5 * log(variance) - 0.5 * ((x - mean)^2 / variance)
            # (We drop the -0.5 * log(2*pi) constant as it's the same for all classes)
            
            term1 = math.log(var)
            term2 = ((x - mean) ** 2) / var
            
            log_prob -= 0.5 * (term1 + term2)
            
        # Keep track of the highest probability
        if log_prob > max_log_prob:
            max_log_prob = log_prob
            best_class = c
            
    return best_class

# --- Example Usage on Board ---
# features = [1.2, -0.5, 3.4, ...] # The 39 MFCC features extracted from the mic
# result_idx = predict_audio_class(features)
# print("Predicted Word:", model_data.CLASSES[result_idx])


# --- Example Usage on Board ---
# ['on', 'off', 'stop', 'uknown']
i = 16
# features = [-2.5686844e+02,  1.3959177e+02,  8.0665617e+00, -4.1448826e+01, -2.9944046e+01,  3.2988396e-01,  6.2178426e+00,  9.0643892e+00, -6.5838027e+00,  5.9275532e+00, -2.8192373e+01,  1.9626240e+01, -1.0130860e+00, -2.5470102e+02,  1.6171916e+02, -9.4717093e+00, -1.6660915e+01,  9.5316715e+00,  1.5823692e+01, -4.0831656e+00, -1.6109245e+01, -1.6403475e+01, -2.7065601e+00, -7.6615257e+00, 1.2409652e+01, -2.4190521e+01, -4.9443311e+02,  8.9877983e+01, 1.8117041e+01,  3.1036968e+01,  5.0818684e+01,  3.1483164e+01, 4.5603323e+00, -1.2117879e+01, -1.4944232e+01,  2.7241716e+00, 7.9998846e+00, -1.2753278e+01, -1.6924435e+01] # 13 features from mic
import random
for j in range(10):
    i = random.randint(0, len(y_test))
    features = X_test[i]
    result_idx = predict_audio_class(features)
    print("Predicted Word:", model_data.CLASSES[result_idx], "and actuall is:", y_test[i] + 1)

Predicted Word: on and actuall is: 1
Predicted Word: unknown and actuall is: 2
Predicted Word: unknown and actuall is: 4
Predicted Word: on and actuall is: 1
Predicted Word: stop and actuall is: 3
Predicted Word: on and actuall is: 1
Predicted Word: unknown and actuall is: 4
Predicted Word: stop and actuall is: 2
Predicted Word: unknown and actuall is: 4
Predicted Word: unknown and actuall is: 1


*LDA + Logistic regression*

In [90]:
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import os

DATA_PATH = '../Data/'
OUTPUT_PATH = '../Output'

# 1. Load the FULL 39-feature dataset
X = np.load(os.path.join(DATA_PATH, 'X_features.npy'))
y = np.load(os.path.join(DATA_PATH, 'y_labels.npy'))

print(f"Loaded Data -> X shape: {X.shape}, y shape: {y.shape}")

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. STAGE 1: Dimensionality Reduction using LDA
print("\n--- Running LDA to reduce 39 features to 3 Super-Features ---")
# n_components is always (number of classes - 1) for LDA
lda = LinearDiscriminantAnalysis(n_components=3)
X_train_lda = lda.fit_transform(X_train, y_train)
X_test_lda = lda.transform(X_test)

print(f"New Training Data Shape after LDA: {X_train_lda.shape}")

# 3. STAGE 2: Optuna Optimization for Logistic Regression
print("\n--- Starting Optuna optimization for Logistic Regression ---")
def objective(trial):
    # Tune the regularization parameter C
    c_val = trial.suggest_float('C', 1e-4, 1e2, log=True)
    
    lr = LogisticRegression(C=c_val, max_iter=5000, random_state=42)
    lr.fit(X_train_lda, y_train)
    preds = lr.predict(X_test_lda)
    
    return accuracy_score(y_test, preds)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print(f"\nBest Optuna Parameters: {study.best_params}")
print(f"Best Accuracy during CV: {study.best_value * 100:.2f}%")

# 4. TRAIN FINAL CLASSIFIER
print("\n--- Training Final Lightweight Classifier ---")
final_lr = LogisticRegression(**study.best_params, max_iter=5000, random_state=42)
final_lr.fit(X_train_lda, y_train)

preds = final_lr.predict(X_test_lda)
unique_classes = np.unique(y_test)
target_names = ['on', 'off', 'stop', 'unknown'][:len(unique_classes)]

print("\nFinal Classification Report:")
print(classification_report(y_test, preds, target_names=target_names))

# 5. EXPORT PURE PYTHON ARRAYS FOR EDGE INFERENCE
print("\n--- Exporting LDA and Classifier Parameters to model_data_lr.py ---")

# Extract LDA parameters
lda_xbar = lda.xbar_.tolist()         # Shape: (39,) -> The mean of each feature
lda_scalings = lda.scalings_.tolist() # Shape: (39, 3) -> The transformation matrix

# Extract Logistic Regression parameters
lr_coef = final_lr.coef_.tolist()           # Shape: (4, 3) -> Weights for the 3 super-features
lr_intercept = final_lr.intercept_.tolist() # Shape: (4,) -> Bias for each class

output_file = os.path.join(OUTPUT_PATH, 'model_data_lr.py')
with open(output_file, 'w') as f:
    f.write("# Auto-Generated LDA + Logistic Regression Export\n\n")
    f.write(f"CLASSES = {target_names}\n\n")
    
    # Write LDA arrays
    f.write("LDA_XBAR = [\n")
    f.write(f"    {[round(float(val), 6) for val in lda_xbar]}\n")
    f.write("]\n\n")
    
    f.write("LDA_SCALINGS = [\n")
    for row in lda_scalings:
        f.write(f"    {[round(float(val), 6) for val in row]},\n")
    f.write("]\n\n")
    
    # Write LR arrays
    f.write("LR_COEF = [\n")
    for class_weights in lr_coef:
        f.write(f"    {[round(float(w), 6) for w in class_weights]},\n")
    f.write("]\n\n")
    
    f.write("LR_INTERCEPT = [\n")
    f.write(f"    {[round(float(i), 6) for i in lr_intercept]}\n")
    f.write("]\n")

print(f"Pipeline successfully exported to: {output_file}")

[I 2026-06-28 19:26:52,559] A new study created in memory with name: no-name-b2e6fe6c-682d-43c4-bf3d-c2723529d4de
[I 2026-06-28 19:26:52,575] Trial 0 finished with value: 0.7773144286905754 and parameters: {'C': 13.25854094375775}. Best is trial 0 with value: 0.7773144286905754.
[I 2026-06-28 19:26:52,589] Trial 1 finished with value: 0.7781484570475397 and parameters: {'C': 0.992236238684586}. Best is trial 1 with value: 0.7781484570475397.
[I 2026-06-28 19:26:52,604] Trial 2 finished with value: 0.7773144286905754 and parameters: {'C': 39.317554872211254}. Best is trial 1 with value: 0.7781484570475397.
[I 2026-06-28 19:26:52,619] Trial 3 finished with value: 0.7773144286905754 and parameters: {'C': 4.578735294040256}. Best is trial 1 with value: 0.7781484570475397.
[I 2026-06-28 19:26:52,633] Trial 4 finished with value: 0.7789824854045038 and parameters: {'C': 0.10661781375213511}. Best is trial 4 with value: 0.7789824854045038.
[I 2026-06-28 19:26:52,649] Trial 5 finished with val

Loaded Data -> X shape: (5995, 39), y shape: (5995,)

--- Running LDA to reduce 39 features to 3 Super-Features ---
New Training Data Shape after LDA: (4796, 3)

--- Starting Optuna optimization for Logistic Regression ---


[I 2026-06-28 19:26:52,738] Trial 11 finished with value: 0.780650542118432 and parameters: {'C': 0.00016268638266783752}. Best is trial 8 with value: 0.7823185988323603.
[I 2026-06-28 19:26:52,749] Trial 12 finished with value: 0.7814845704753962 and parameters: {'C': 0.00024340729059622702}. Best is trial 8 with value: 0.7823185988323603.
[I 2026-06-28 19:26:52,762] Trial 13 finished with value: 0.7848206839032527 and parameters: {'C': 0.0029286790910135756}. Best is trial 13 with value: 0.7848206839032527.
[I 2026-06-28 19:26:52,777] Trial 14 finished with value: 0.7823185988323603 and parameters: {'C': 0.004492264036589847}. Best is trial 13 with value: 0.7848206839032527.
[I 2026-06-28 19:26:52,791] Trial 15 finished with value: 0.7823185988323603 and parameters: {'C': 0.004402437769723173}. Best is trial 13 with value: 0.7848206839032527.
[I 2026-06-28 19:26:52,803] Trial 16 finished with value: 0.7848206839032527 and parameters: {'C': 0.0021897825587711174}. Best is trial 13 wit


Best Optuna Parameters: {'C': 0.0001127057422495509}
Best Accuracy during CV: 78.65%

--- Training Final Lightweight Classifier ---

Final Classification Report:
              precision    recall  f1-score   support

          on       0.81      0.83      0.82       299
         off       0.77      0.82      0.79       300
        stop       0.86      0.80      0.83       300
     unknown       0.71      0.70      0.70       300

    accuracy                           0.79      1199
   macro avg       0.79      0.79      0.79      1199
weighted avg       0.79      0.79      0.79      1199


--- Exporting LDA and Classifier Parameters to model_data_lr.py ---
Pipeline successfully exported to: ../Output\model_data_lr.py


In [91]:
import sys
import os

sys.path.append(os.path.abspath('../Output'))

import model_data_lr as model_data

def predict_audio_class(mfcc_features):
    """
    Two-stage inference:
    1. LDA Transform: Reduces 39 features to 3.
    2. Logistic Regression: Predicts the class based on the 3 features.
    """
    num_original_features = len(model_data.LDA_XBAR[0])
    num_super_features = len(model_data.LDA_SCALINGS[0])
    num_classes = len(model_data.CLASSES)
    
    # --- STAGE 1: LDA Transformation ---
    # Step 1A: Center the data (subtract the mean)
    centered_features = [0.0] * num_original_features
    for i in range(num_original_features):
        centered_features[i] = mfcc_features[i] - model_data.LDA_XBAR[0][i]
        
    # Step 1B: Matrix multiplication (Centered Data * Scalings)
    lda_features = [0.0] * num_super_features
    for i in range(num_super_features):
        sum_val = 0.0
        for j in range(num_original_features):
            sum_val += centered_features[j] * model_data.LDA_SCALINGS[j][i]
        lda_features[i] = sum_val
        
    # --- STAGE 2: Logistic Regression ---
    class_scores = [0.0] * num_classes
    
    for c in range(num_classes):
        # Start with the bias/intercept for this class
        score = model_data.LR_INTERCEPT[0][c]
        
        # Dot product: Super-features * Class Weights
        for i in range(num_super_features):
            score += lda_features[i] * model_data.LR_COEF[c][i]
            
        class_scores[c] = score
        
    # Find the index of the highest score (Argmax)
    best_class = 0
    max_score = class_scores[0]
    for i in range(1, num_classes):
        if class_scores[i] > max_score:
            max_score = class_scores[i]
            best_class = i
            
    return best_class

# --- Example Usage on Board ---
# features = [1.2, -0.5, 3.4, ...] # The 39 MFCC features extracted from the mic
# result_idx = predict_audio_class(features)
# print("Predicted Word:", model_data.CLASSES[result_idx])

import random
for j in range(10):
    i = random.randint(0, len(y_test))
    features = X_test[i]
    result_idx = predict_audio_class(features)
    print("Predicted Word:", model_data.CLASSES[result_idx], "and actuall is:", y_test[i] + 1)

Predicted Word: stop and actuall is: 3
Predicted Word: off and actuall is: 2
Predicted Word: off and actuall is: 2
Predicted Word: unknown and actuall is: 1
Predicted Word: unknown and actuall is: 4
Predicted Word: stop and actuall is: 4
Predicted Word: on and actuall is: 1
Predicted Word: off and actuall is: 2
Predicted Word: unknown and actuall is: 4
Predicted Word: stop and actuall is: 3


LDA + Logistic regression
Targeted training

In [95]:
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import os

DATA_PATH = '../Data/'
OUTPUT_PATH = '../Output/'

# Ensure the Output directory exists
os.makedirs(OUTPUT_PATH, exist_ok=True)

# 1. Load the FULL 39-feature dataset
X = np.load(os.path.join(DATA_PATH, 'X_features.npy'))
y = np.load(os.path.join(DATA_PATH, 'y_labels.npy'))

print(f"Loaded Data -> X shape: {X.shape}, y shape: {y.shape}")

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. STAGE 1 & 2: Optuna Optimization for LDA & Logistic Regression combined
print("\n--- Starting Optuna optimization with Focus on Targets ---")
def objective(trial):
    # --- LDA PARAMETERS ---
    lda_shrinkage = trial.suggest_float('lda_shrinkage', 0.0, 1.0)
    lda = LinearDiscriminantAnalysis(solver='eigen', shrinkage=lda_shrinkage, n_components=3)
    
    X_train_lda = lda.fit_transform(X_train, y_train)
    X_test_lda = lda.transform(X_test)
    
    # --- LOGISTIC REGRESSION PARAMETERS ---
    c_val = trial.suggest_float('C', 1e-4, 1e2, log=True)
    
    # --- CLASS PRIORITIZATION ---
    target_weight = trial.suggest_float('target_weight', 1.0, 5.0)
    class_weights = {0: target_weight, 1: target_weight, 2: target_weight, 3: 1.0}
    
    lr = LogisticRegression(C=c_val, class_weight=class_weights, max_iter=5000, random_state=42)
    lr.fit(X_train_lda, y_train)
    preds = lr.predict(X_test_lda)
    
    return accuracy_score(y_test, preds)

# Run optimization
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print(f"\nBest Optuna Parameters: {study.best_params}")
print(f"Best Accuracy during CV: {study.best_value * 100:.2f}%")

# 3. TRAIN FINAL PIPELINE WITH BEST PARAMETERS
print("\n--- Training Final Lightweight & Focused Classifier ---")

best_shrinkage = study.best_params['lda_shrinkage']
best_C = study.best_params['C']
best_target_weight = study.best_params['target_weight']

# Train Final LDA (using 'eigen')
final_lda = LinearDiscriminantAnalysis(solver='eigen', shrinkage=best_shrinkage, n_components=3)
X_train_lda_final = final_lda.fit_transform(X_train, y_train)
X_test_lda_final = final_lda.transform(X_test)

# Train Final Logistic Regression
final_weights = {0: best_target_weight, 1: best_target_weight, 2: best_target_weight, 3: 1.0}
final_lr = LogisticRegression(C=best_C, class_weight=final_weights, max_iter=5000, random_state=42)
final_lr.fit(X_train_lda_final, y_train)

# Evaluate
preds = final_lr.predict(X_test_lda_final)
unique_classes = np.unique(y_test)
target_names = ['on', 'off', 'stop', 'unknown'][:len(unique_classes)]

print("\nFinal Classification Report:")
print(classification_report(y_test, preds, target_names=target_names))

# 4. EXPORT PURE PYTHON ARRAYS FOR EDGE INFERENCE
print("\n--- Exporting LDA and Classifier Parameters ---")

# Extract LDA parameters (No xbar_ needed for 'eigen' solver!)
lda_scalings = final_lda.scalings_.tolist() 

# Extract Logistic Regression parameters
lr_coef = final_lr.coef_.tolist()           
lr_intercept = final_lr.intercept_.tolist() 

output_file = os.path.join(OUTPUT_PATH, 'model_data_lr.py')
with open(output_file, 'w') as f:
    f.write("# Auto-Generated Robust LDA (Eigen) + Logistic Regression Export\n\n")
    f.write(f"CLASSES = {target_names}\n\n")
    
    f.write("LDA_SCALINGS = [\n")
    for row in lda_scalings:
        f.write(f"    {[round(float(val), 6) for val in row]},\n")
    f.write("]\n\n")
    
    f.write("LR_COEF = [\n")
    for class_weights in lr_coef:
        f.write(f"    {[round(float(w), 6) for w in class_weights]},\n")
    f.write("]\n\n")
    
    f.write("LR_INTERCEPT = [\n")
    f.write(f"    {[round(float(i), 6) for i in lr_intercept]}\n")
    f.write("]\n")

print(f"Pipeline successfully exported to: {output_file}")

[I 2026-06-28 19:38:26,857] A new study created in memory with name: no-name-150b1f20-7adc-47be-b6d4-99554a7b0f6e


Loaded Data -> X shape: (5995, 39), y shape: (5995,)

--- Starting Optuna optimization with Focus on Targets ---


[I 2026-06-28 19:38:27,081] Trial 0 finished with value: 0.6922435362802335 and parameters: {'lda_shrinkage': 0.7938982354846156, 'C': 0.07689260384397603, 'target_weight': 2.1716380662385486}. Best is trial 0 with value: 0.6922435362802335.
[I 2026-06-28 19:38:27,173] Trial 1 finished with value: 0.700583819849875 and parameters: {'lda_shrinkage': 0.9353299310062916, 'C': 11.264586594125822, 'target_weight': 1.019716381552148}. Best is trial 1 with value: 0.700583819849875.
[I 2026-06-28 19:38:27,204] Trial 2 finished with value: 0.7289407839866555 and parameters: {'lda_shrinkage': 0.06114779280482585, 'C': 0.19948095014170805, 'target_weight': 4.876951294291059}. Best is trial 2 with value: 0.7289407839866555.
[I 2026-06-28 19:38:27,238] Trial 3 finished with value: 0.7781484570475397 and parameters: {'lda_shrinkage': 0.006291972748786456, 'C': 0.004083072596120302, 'target_weight': 1.588743500510926}. Best is trial 3 with value: 0.7781484570475397.
[I 2026-06-28 19:38:27,275] Trial 


Best Optuna Parameters: {'lda_shrinkage': 0.06303600198532323, 'C': 0.8905175026288367, 'target_weight': 1.0017510110151215}
Best Accuracy during CV: 78.32%

--- Training Final Lightweight & Focused Classifier ---

Final Classification Report:
              precision    recall  f1-score   support

          on       0.81      0.81      0.81       299
         off       0.80      0.79      0.80       300
        stop       0.87      0.78      0.82       300
     unknown       0.67      0.75      0.71       300

    accuracy                           0.78      1199
   macro avg       0.79      0.78      0.78      1199
weighted avg       0.79      0.78      0.78      1199


--- Exporting LDA and Classifier Parameters ---
Pipeline successfully exported to: ../Output/model_data_lr.py


In [97]:
import model_data_lr

def predict_audio_class(mfcc_features):
    """
    Super-optimized Inference Engine for MicroPython (LDA Eigen + Logistic Regression).
    File size: < 3KB | Execution time: < 2ms
    """
    num_original_features = 39 # 13 MFCCs * 3 Time Frames
    num_super_features = 3     # Reduced dimensions by LDA
    num_classes = len(model_data_lr.CLASSES)
    
    # --- STAGE 1: LDA Transformation (Direct Matrix Multiplication) ---
    # Since we used 'eigen' solver, we directly multiply raw features by SCALINGS
    lda_features = [0.0] * num_super_features
    for i in range(num_super_features):
        sum_val = 0.0
        for j in range(num_original_features):
            sum_val += mfcc_features[j] * model_data_lr.LDA_SCALINGS[j][i]
        lda_features[i] = sum_val
        
    # --- STAGE 2: Logistic Regression Inference ---
    class_scores = [0.0] * num_classes
    for c in range(num_classes):
        # Start with the bias/intercept for this class
        score = model_data_lr.LR_INTERCEPT[0][c]
        
        # Dot product: Super-features * Class Weights
        for i in range(num_super_features):
            score += lda_features[i] * model_data_lr.LR_COEF[c][i]
            
        class_scores[c] = score
        
    # Find the index of the highest score (Argmax)
    best_class = 0
    max_score = class_scores[0]
    for i in range(1, num_classes):
        if class_scores[i] > max_score:
            max_score = class_scores[i]
            best_class = i
            
    return best_class

# --- Example Usage on Board ---
# features = [1.2, -0.5, 3.4, ...] # 39 features extracted from mic (VAD applied)
# result_idx = predict_audio_class(features)
# print("Predicted Word:", model_data_lr.CLASSES[result_idx])

import random
for j in range(10):
    i = random.randint(0, len(y_test))
    features = X_test[i]
    result_idx = predict_audio_class(features)
    print("Predicted Word:", model_data.CLASSES[result_idx], "and actuall is:", y_test[i] + 1)

Predicted Word: on and actuall is: 2
Predicted Word: on and actuall is: 1
Predicted Word: unknown and actuall is: 4
Predicted Word: on and actuall is: 2
Predicted Word: on and actuall is: 4
Predicted Word: on and actuall is: 1
Predicted Word: stop and actuall is: 4
Predicted Word: on and actuall is: 1
Predicted Word: on and actuall is: 1
Predicted Word: on and actuall is: 2
